# Social Media Sentiment Analysis for Brand Monitoring

## 1. Problem Definition & Dataset Selection

### Domain of Interest:
The project focuses on **social media sentiment analysis** to monitor brand perception, a key area in digital marketing and customer relationship management.

### Problem Statement:
With the rise of social media, brands need to understand customer sentiment to manage reputation and engagement. This project aims to classify social media posts as positive, negative, or neutral using TextBlob and provide real-time insights via a Streamlit app.

### Dataset:
A dataset of 50,000 social media posts stored in 'synthetic_social_media_dataset.csv', containing features: `Post_Text`, `Likes_Count`, `Shares_Count`, `Comment_Count`, `Post_Time`, `Sentiment_Label`, `Hashtags`, `Platform`, `Post_Length`.

## 2. Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from wordcloud import WordCloud
from collections import Counter
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import joblib
import re

## 3. Loading Dataset

Load the external dataset from 'synthetic_social_media_dataset.csv'.

In [ ]:
dataset = pd.read_csv('synthetic_social_media_dataset.csv')
dataset.head()

## 4. Data Preprocessing

- Clean text by removing URLs, hashtags, mentions.
- Tokenize and lemmatize text using TextBlob.
- Handle missing values and outliers in engagement metrics.

In [ ]:
def clean_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) 
    text = re.sub(r'#\w+|@\w+', '', text) 
    text = text.lower().strip()
    return text

dataset['Cleaned_Text'] = dataset['Post_Text'].apply(clean_text)

def get_sentiment(text):
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    if polarity > 0.1:  
        return 'Positive'
    elif polarity < -0.1:  
        return 'Negative'
    else:
        return 'Neutral'

dataset['Predicted_Sentiment'] = dataset['Cleaned_Text'].apply(get_sentiment)


dataset = dataset.dropna()


for col in ['Likes_Count', 'Shares_Count', 'Comment_Count']:
    Q1 = dataset[col].quantile(0.25)
    Q3 = dataset[col].quantile(0.75)
    IQR = Q3 - Q1
    dataset = dataset[(dataset[col] >= Q1 - 1.5 * IQR) & (dataset[col] <= Q3 + 1.5 * IQR)]

## 5. Exploratory Data Analysis

- Visualize sentiment distribution by platform.
- Analyze engagement metrics (likes, shares, comments) by sentiment.
- Examine post length vs. sentiment.
- Generate word clouds for each sentiment.

In [ ]:

dataset['Post_Time'] = pd.to_datetime(dataset['Post_Time'])

# 5.1 Sentiment distribution by platform
plt.figure(figsize=(8, 5))
sns.countplot(x='Sentiment_Label', hue='Platform', data=dataset)
plt.title('Sentiment Distribution by Platform')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.savefig('sentiment_by_platform.png')
plt.show()

# 5.2 Engagement metrics by sentiment
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.boxplot(x='Sentiment_Label', y='Likes_Count', data=dataset, ax=axes[0])
axes[0].set_title('Likes by Sentiment')
sns.boxplot(x='Sentiment_Label', y='Shares_Count', data=dataset, ax=axes[1])
axes[1].set_title('Shares by Sentiment')
sns.boxplot(x='Sentiment_Label', y='Comment_Count', data=dataset, ax=axes[2])
axes[2].set_title('Comments by Sentiment')
plt.tight_layout()
plt.savefig('engagement_by_sentiment.png')
plt.show()

# 5.3 Post length vs. sentiment
plt.figure(figsize=(8, 5))
sns.boxplot(x='Sentiment_Label', y='Post_Length', data=dataset)
plt.title('Post Length by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Post Length (Characters)')
plt.savefig('post_length_by_sentiment.png')
plt.show()

# 5.4 Word clouds for each sentiment
for sentiment in ['Positive', 'Negative', 'Neutral']:
    text = ' '.join(dataset[dataset['Sentiment_Label'] == sentiment]['Cleaned_Text'])
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud for {sentiment} Posts')
    plt.savefig(f'wordcloud_{sentiment.lower()}.png')
    plt.show()

# 5.5 Sentiment trends over time (monthly)
dataset['Month'] = dataset['Post_Time'].dt.to_period('M').astype(str)
plt.figure(figsize=(12, 6))
sns.countplot(x='Month', hue='Sentiment_Label', data=dataset)
plt.title('Sentiment Trends Over Time (Monthly)')
plt.xlabel('Month')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.savefig('sentiment_trends_over_time.png')
plt.show()

# 5.6  Top 5 hashtags by sentiment (fixed for varying hashtag counts)
def extract_hashtags(text):
    if isinstance(text, str):  # Ensure text is a string
        return re.findall(r'#\w+', text.lower())
    return []

dataset['Hashtag_List'] = dataset['Hashtags'].apply(extract_hashtags)
hashtag_freq = {}
for sentiment in ['Positive', 'Negative', 'Neutral']:
    hashtags = [htag for tags in dataset[dataset['Sentiment_Label'] == sentiment]['Hashtag_List'] for htag in tags]
   
    top_hashtags = Counter(hashtags).most_common(5)
    if len(top_hashtags) < 5:
        top_hashtags.extend([('', 0)] * (5 - len(top_hashtags)))
    hashtag_freq[sentiment] = top_hashtags
    print(f"{sentiment}: {len(hashtags)} total hashtags, top 5: {top_hashtags}")

hashtag_df = pd.DataFrame({
    'Hashtag': [htag[0] for sentiment in hashtag_freq for htag in hashtag_freq[sentiment]],
    'Count': [htag[1] for sentiment in hashtag_freq for htag in hashtag_freq[sentiment]],
    'Sentiment': [sentiment for sentiment in hashtag_freq for _ in range(5)]
})


print(f"Hashtag list length: {len([htag[0] for sentiment in hashtag_freq for htag in hashtag_freq[sentiment]])}")
print(f"Count list length: {len([htag[1] for sentiment in hashtag_freq for htag in hashtag_freq[sentiment]])}")
print(f"Sentiment list length: {len([sentiment for sentiment in hashtag_freq for _ in range(5)])}")

plt.figure(figsize=(10, 6))
sns.barplot(x='Count', y='Hashtag', hue='Sentiment', data=hashtag_df)
plt.title('Top 5 Hashtags by Sentiment')
plt.xlabel('Frequency')
plt.ylabel('Hashtag')
plt.savefig('top_hashtags_by_sentiment.png')
plt.show()

# 5.7  Correlation heatmap of engagement metrics
plt.figure(figsize=(8, 6))
corr = dataset[['Likes_Count', 'Shares_Count', 'Comment_Count', 'Post_Length']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Engagement Metrics and Post Length')
plt.savefig('engagement_correlation_heatmap.png')
plt.show()

# 5.8  Platform-specific engagement distributions
fig, axes = plt.subplots(3, 1, figsize=(10, 12))
sns.histplot(data=dataset, x='Likes_Count', hue='Platform', multiple='stack', ax=axes[0])
axes[0].set_title('Likes Distribution by Platform')
sns.histplot(data=dataset, x='Shares_Count', hue='Platform', multiple='stack', ax=axes[1])
axes[1].set_title('Shares Distribution by Platform')
sns.histplot(data=dataset, x='Comment_Count', hue='Platform', multiple='stack', ax=axes[2])
axes[2].set_title('Comments Distribution by Platform')
plt.tight_layout()
plt.savefig('platform_engagement_distributions.png')
plt.show()

# 5.9  Sentiment distribution by post length bins
dataset['Length_Bin'] = pd.cut(dataset['Post_Length'], bins=[0, 50, 100, 150, 200, 250, float('inf')],
                               labels=['0–50', '51–100', '101–150', '151–200', '201–250', '250+'])
plt.figure(figsize=(10, 6))
sns.countplot(x='Length_Bin', hue='Sentiment_Label', data=dataset)
plt.title('Sentiment Distribution by Post Length Bins')
plt.xlabel('Post Length Bin (Characters)')
plt.ylabel('Count')
plt.savefig('sentiment_by_length_bins.png')
plt.show()

## 6. Unsupervised Learning

- Apply PCA to visualize text features.
- Use K-Means for clustering.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=100)
X_tfidf = tfidf.fit_transform(dataset['Cleaned_Text']).toarray()

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_tfidf)

plt.figure(figsize=(8, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=dataset['Sentiment_Label'].map({'Positive': 0, 'Negative': 1, 'Neutral': 2}))
plt.title('PCA Visualization of Text Features')
plt.savefig('pca_visualization.png')
plt.show()

kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_tfidf)
dataset['Cluster'] = clusters

## 7. Model Evaluation

- Evaluate TextBlob predictions against ground truth.

In [ ]:
print(classification_report(dataset['Sentiment_Label'], dataset['Predicted_Sentiment']))

## 8. Model Deployment

- Save preprocessing pipeline and model.
- Test with sample input.

## 9. Project Summary

- Developed a sentiment analysis system using TextBlob.
- Preprocessed text data and visualized sentiment trends.
- Evaluated model performance and deployed via Streamlit.